# 01 · Análise Exploratória (EDA)

Objetivo: entender volume, qualidade e distribuição das features e do target.

> As funções de limpeza vêm de `src/churn/data.py`. Ajustes descobertos aqui devem ser levados para lá (não duplicar lógica no notebook).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from churn.config import CATEGORICAL_FEATURES, NUMERIC_FEATURES, SEED, TARGET
from churn.data import clean, load_raw

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Volume e tipos

In [ ]:
raw = load_raw()
print(raw.shape)
raw.info()
raw.head()

## 2. Qualidade dos dados

- Nulos e brancos (atenção a `TotalCharges`)
- Duplicados
- Cardinalidade das categóricas

In [ ]:
print(raw.isna().sum().sum(), 'nulos')
print((raw['TotalCharges'].str.strip() == '').sum(), 'TotalCharges em branco')
print(raw.duplicated(subset='customerID').sum(), 'IDs duplicados')
raw[CATEGORICAL_FEATURES].nunique()

In [ ]:
df = clean(raw)
df.describe()

## 3. Distribuição do target

In [ ]:
df[TARGET].value_counts(normalize=True).rename('proporção')

## 4. Numéricas vs. churn

In [ ]:
fig, axes = plt.subplots(1, len(NUMERIC_FEATURES), figsize=(15, 4))
for ax, col in zip(axes, NUMERIC_FEATURES):
    sns.kdeplot(data=df, x=col, hue=TARGET, common_norm=False, fill=True, ax=ax)
plt.tight_layout()

## 5. Categóricas vs. taxa de churn

In [ ]:
for col in CATEGORICAL_FEATURES:
    rate = df.groupby(col)[TARGET].agg(['mean', 'size']).rename(columns={'mean': 'churn_rate'})
    display(rate.sort_values('churn_rate', ascending=False))

## 6. Correlações numéricas

In [ ]:
sns.heatmap(df[NUMERIC_FEATURES + [TARGET]].corr(), annot=True, cmap='RdBu_r', center=0)

## 7. Conclusões

- _Taxa de churn e implicação (desbalanceamento → métricas e class_weight)_
- _Principais sinais (contrato mensal, tenure baixo, fibra, electronic check...)_
- _Problemas de qualidade encontrados e como foram tratados_